# Chapter 2 · Language Modeling

A name generator that learns by counting which letter follows each letter. Standard Python and a CPU: there is nothing to install. Run the cells in order.

The full explanation is in the PDF chapter. This notebook is the lab: the same program, split into parts, so you can run it and experiment.

## 1. The data

Four training names, two held-out names that are not counted, and two marks: `^` for the start and `$` for the end. The possible outputs are the four letters and the end.

In [1]:
"""Chapter 2: character bigrams. Standard library and CPU only."""
from collections import Counter
import math
import random

TRAINING = ("ana", "anna", "alba", "alan")
HELD_OUT = ("lana", "bala")
START, END = "^", "$"
LETTERS = tuple(sorted(set("".join(TRAINING))))
OUTPUTS = LETTERS + (END,)
CONTEXTS = (START,) + LETTERS

In [2]:
print("Outputs:", OUTPUTS)
print("Contexts:", CONTEXTS)

Outputs: ('a', 'b', 'l', 'n', '$')
Contexts: ('^', 'a', 'b', 'l', 'n')


## 2. Counting

`pairs` walks through a name in pairs, with its marks; `count` adds one to the cell of each pair. Before running it, predict the `a` row and the total number of transitions.

In [3]:
def pairs(name):
    string = START + name + END
    return zip(string, string[1:])


def count(names):
    counts = {c: Counter() for c in CONTEXTS}
    for name in names:
        if not name or any(c not in LETTERS for c in name):
            raise ValueError("Empty name or letter outside the vocabulary")
        for previous, following in pairs(name):
            counts[previous][following] += 1
    return counts

In [4]:
print("Pairs of ana:", list(pairs("ana")))
counts = count(TRAINING)
for c in CONTEXTS:
    print(c, [counts[c][s] for s in OUTPUTS])
print("Total:", sum(sum(row.values()) for row in counts.values()), "transitions")

Pairs of ana: [('^', 'a'), ('a', 'n'), ('n', 'a'), ('a', '$')]
^ [4, 0, 0, 0, 0]
a [0, 0, 2, 3, 3]
b [1, 0, 0, 0, 0]
l [1, 1, 0, 0, 0]
n [2, 0, 0, 1, 1]
Total: 19 transitions


## 3. From counts to probabilities

`probabilities` divides each row by its total. With `alpha` it adds one imaginary slip of each output: that is smoothing. Why are there thirteen slips after `a`?

In [5]:
def probabilities(counts, alpha=1.0):
    if alpha < 0:
        raise ValueError("alpha must be non-negative")
    table = {}
    for c in CONTEXTS:
        total = sum(counts[c].values()) + alpha * len(OUTPUTS)
        if total == 0:
            raise ValueError("Empty row: use smoothing")
        table[c] = {s: (counts[c][s] + alpha) / total
                    for s in OUTPUTS}
    return table

In [6]:
unsmoothed = probabilities(counts, alpha=0)
table = probabilities(counts, alpha=1)
print("Unsmoothed a row:", [round(unsmoothed["a"][s], 3) for s in OUTPUTS])
print("Smoothed a row:  ", [round(table["a"][s], 3) for s in OUTPUTS])
print("The smoothed row adds up to:", sum(table["a"].values()))

Unsmoothed a row: [0.0, 0.0, 0.25, 0.375, 0.375]
Smoothed a row:   [0.077, 0.077, 0.231, 0.308, 0.308]
The smoothed row adds up to: 1.0


## 4. From cost to loss

The cost of a transition is −ln p: zero if the model gave probability 1 to what appeared, and larger the less likely the model considered it. Python writes the first cost as `-0.0`, a signed zero.

In [7]:
import math

for p in (1.0, 0.5, 0.1):
    cost = -math.log(p)
    print(p, round(cost, 3))

1.0 -0.0
0.5 0.693
0.1 2.303


`loss` averages the costs of all the transitions of the texts it is given. It does not sample anything: the output is already given by the text.

In [8]:
def loss(table, names):
    costs = []
    for name in names:
        if not name or any(c not in LETTERS for c in name):
            raise ValueError("Empty name or letter outside the vocabulary")
        for c, s in pairs(name):
            p = table[c][s]
            if p == 0:
                return math.inf
            costs.append(-math.log(p))
    if not costs:
        raise ValueError("No transitions to evaluate")
    return sum(costs) / len(costs)

Follow the paths of `ana` and `lana`: the probability of a path is the product of its transitions, and the NLL is the mean of their costs.

In [9]:
for name in ("ana", "lana"):
    ps = [table[c][s] for c, s in pairs(name)]
    print(name, [round(p, 3) for p in ps])
    print("   path probability:", round(math.prod(ps), 6), " mean NLL:", round(loss(table, [name]), 6))

ana [0.556, 0.308, 0.333, 0.308]
   path probability: 0.017532  mean NLL: 1.010927
lana [0.111, 0.286, 0.308, 0.333, 0.308]
   path probability: 0.001002  mean NLL: 1.381182


## 5. Generating

`generate` samples an output from the current row, adds it, and uses it as the new context, until it draws the end or uses up twenty decisions.

In [10]:
def generate(table, rng, max_steps=20):
    """Return (text, True if it ended with END). The limit adds no END."""
    c, letters = START, []
    for _ in range(max_steps):
        weights = [table[c][s] for s in OUTPUTS]
        s = rng.choices(OUTPUTS, weights=weights, k=1)[0]
        if s == END:
            return "".join(letters), True
        letters.append(s)
        c = s
    return "".join(letters), False

## 6. The complete experiment

`run` counts, compares with the uniform model, evaluates with and without smoothing, and generates ten samples with seed 7.

In [11]:
def run():
    counts = count(TRAINING)
    table = probabilities(counts, alpha=1.0)
    uniform = {c: {s: 1 / len(OUTPUTS) for s in OUTPUTS}
               for c in CONTEXTS}
    print("Columns:", OUTPUTS)
    for c in CONTEXTS:
        print(c, [counts[c][s] for s in OUTPUTS])
    print("Uniform NLL: %.6f" % loss(uniform, HELD_OUT))
    print("Training NLL: %.6f" % loss(table, TRAINING))
    print("Held-out NLL: %.6f" % loss(table, HELD_OUT))
    unsmoothed = probabilities(counts, alpha=0)
    print("Unsmoothed training NLL: %.6f"
          % loss(unsmoothed, TRAINING))
    print("Unsmoothed held-out NLL:", loss(unsmoothed, HELD_OUT))
    rng = random.Random(7)
    for _ in range(10):
        text, ended = generate(table, rng)
        status = "end" if ended else "truncated"
        print(repr(text), status)

In [12]:
run()

Columns: ('a', 'b', 'l', 'n', '$')
^ [4, 0, 0, 0, 0]
a [0, 0, 2, 3, 3]
b [1, 0, 0, 0, 0]
l [1, 1, 0, 0, 0]
n [2, 0, 0, 1, 1]
Uniform NLL: 1.609438
Training NLL: 1.113961
Held-out NLL: 1.409950
Unsmoothed training NLL: 0.747513
Unsmoothed held-out NLL: inf
'ablanbananabbnall' end
'bb' end
'a' end
'abalnannblaallbanlbn' truncated
'lanl' end
'lb' end
'annanannn' end
'a' end
'blb' end
'' end


## 7. Experiment

Change the seed and run it again (exercise 9): what changes and what does not? Then try other values of `alpha` (exercise 12) and look at the `a` row and the training loss. Write down first what you expect to happen, and do not use the held-out set to decide.

In [13]:
seed = 7  # try others
rng = random.Random(seed)
for _ in range(10):
    text, ended = generate(table, rng)
    print(repr(text), "end" if ended else "truncated")

'ablanbananabbnall' end
'bb' end
'a' end
'abalnannblaallbanlbn' truncated
'lanl' end
'lb' end
'annanannn' end
'a' end
'blb' end
'' end


In [14]:
alpha = 0.1  # also try 1 and 10
trial = probabilities(counts, alpha=alpha)
print("a row:", [round(trial["a"][s], 3) for s in OUTPUTS])
print("Training loss:", round(loss(trial, TRAINING), 3))

a row: [0.012, 0.012, 0.247, 0.365, 0.365]
Training loss: 0.817


## Exercises

1. Write the five transitions of `alba`, including the start and the end. Explain why there is one more transition than there are letters.

2. Apply the greedy choice to the smoothed table, starting at the start and breaking ties in favor of the output that comes first in the order `a`, `b`, `l`, `n`, end. What string do you get? Does it end?

3. Calculate, without smoothing, the probability of `l` after `a` and of `a` after `l`. Explain why they do not have to be equal.

4. With α=1, write the five probabilities after `n` and check their sum.

5. Calculate the smoothed probability of generating an empty string. Should the program hide it in order to evaluate the samples honestly?

6. Follow the complete path of `anna` with the smoothed table: calculate its probability and its mean NLL, including the end.

7. With the smoothed table, calculate by hand the probability of the path of `lana`. Identify the transition that receives probability zero without smoothing.

8. Explain why the uniform model has mean loss ln 5 on any valid string. Would it change if we used base-two logarithms?

9. Change only the seed. Compare samples and losses, and explain which part of the procedure has changed.

10. Explain why we cannot evaluate `rosa` with this table. Distinguish an unknown letter from an unobserved pair.

11. Compare the context used after `an` and after `alan`. What information would a model need to tell them apart? (Chapter 4 will do it.)

12. Try `alpha=0.1` and `alpha=10`, in the notebook or by changing line 72 of the program. Describe what happens to the `a` row and to the training loss. Do not use the held-out set to decide which one is better.

13. Write a five-line conclusion about the comparison with the uniform model: mention the held-out set, the number of transitions, the limitations of the data and the difference between the score and the quality of the samples.

## References

Teaching idea: the first lesson of [makemore](https://www.youtube.com/watch?v=PaCmpygFfXo), part of [Zero to Hero](https://github.com/karpathy/nn-zero-to-hero), by Andrej Karpathy. The minimal collection, the code, and the explanations are original.